## Creates patterns and semantic_annotations tables

all annotated verb patterns

In [1]:
import sqlite3
import pandas as pd
import os
import numpy as np
import re
from estnltk import Text

## Configuration

In [2]:
# mustrite salvestamise andmebaas
VERB_PATTERN_DB = "../example_data/verb_patterns.db"

MAARUS_STATUS = 'alati' #'mitte kunagi'
MAARUS_STATUS2 = MAARUS_STATUS.replace(" ", "_")

# deprel mustritele
DEPREL = 'obl'

# lõpptabel, va ühes cellis, kus on vaja käsitsi muuta
PATTERN_TABLE_NAME = f"patterns" 

SEMANTIC_ANNOTATIONS_TABLE = "semantic_annotations"

ANNOTATION_TABLE = "../source_data/every_verb_case_obl.csv"

## Create and connect to db

In [3]:
# verbimustrite andmebaasi loomine/ühendumine
con = sqlite3.connect(VERB_PATTERN_DB)
cur = con.cursor()

In [4]:
# ainult esmakordseks db faili loomise ajaks
cur.execute('pragma encoding=UTF8') 

## Workflow

### Read in and process verb annotations

In [5]:
# märgendused
margendused = pd.read_csv(ANNOTATION_TABLE, sep=";", encoding="utf-8")
margendused

,verbobl,verb,case,isikumäärus,aja-kohamäärus,muu
0,saama - abl (kellelt/millelt),saama,abl (kellelt/millelt),vahel,vahel,mitte kunagi
1,tulema - abl (kellelt/millelt),tulema,abl (kellelt/millelt),vahel,vahel,mitte kunagi
2,küsima - abl (kellelt/millelt),küsima,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi
3,nõudma - abl (kellelt/millelt),nõudma,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi
4,võtma - abl (kellelt/millelt),võtma,abl (kellelt/millelt),vahel,vahel,mitte kunagi
...,...,...,...,...,...,...
10574,musitseerima - in (kelles/milles),musitseerima,in (kelles/milles),mitte kunagi,alati,muu
10575,kõigutama - in (kelles/milles),kõigutama,in (kelles/milles),mitte kunagi,mitte kunagi,muu
10576,kätlema - in (kelles/milles),kätlema,in (kelles/milles),mitte kunagi,alati,muu
10577,kõmmutama - in (kelles/milles),kõmmutama,in (kelles/milles),mitte kunagi,alati,mitte kunagi


In [6]:
def process_verbs(df):
    # verbide tükeldamine (pea)verbiks ja selle muudeks osadeks
    verb_word = []
    compound_prt1 = []
    compound_prt2 = []
    compound_prt3 = []

    for idx, row in df.iterrows():
        verb = ''
        compound = ['', '', ''] # in current data, there are max 3 compound pieces
        pieces = row['word'].strip().split()

        j = len(pieces)-1 # (pea)verbi leidmine. Oletus on, et pikema konstruktsiooni viimane verbist liige on peaverb
        while j >= 0:
            text = Text(pieces[j]).tag_layer('morph_analysis')
            if 'V' in text.morph_analysis.partofspeech[0]:
                verb = pieces[j]
                pieces.pop(j)
                break
            j-=1

        for i in range(len(pieces)): # allesjäänud jupid määratakse konstruktsiooni ülejäänud osadeks
            compound[i] = pieces[i]

        verb_word.append(verb)
        compound_prt1.append(compound[0])
        compound_prt2.append(compound[1])
        compound_prt3.append(compound[2])
        
    return verb_word, compound_prt1, compound_prt2, compound_prt3

In [19]:
def process_isikumaarused(maarus_status, deprel):
    
    # salvesta isikumäärused
    isikumaarused = margendused#[margendused["isikumäärus"]==maarus_status]
    isikumaarused = isikumaarused.reset_index().drop("index", axis=1)
    #isikumaarused.to_csv(f"isikumaarused_{maarus_status}.csv", sep=",", encoding="utf-8")
    
    # lüüa lahku juhud kus verbile on antud komaga eraldatuna kaks võimalikku kaassõna 
    # nt olema kokku, võlgu -> olema kokku ja olema võlgu

    uus1 = []
    for i in range(len(isikumaarused)):
        verb = isikumaarused.iloc[i]["verb"].strip()
        case_osad =  isikumaarused.iloc[i]["case"].strip().split(" ")
        if len(case_osad)>=2:
            verbobl = isikumaarused.iloc[i]["verbobl"]
            isik = isikumaarused.iloc[i]["isikumäärus"]
            koht = isikumaarused.iloc[i]["aja-kohamäärus"]
            muu = isikumaarused.iloc[i]["muu"]
            case = case_osad[0].strip()
            gov = case_osad[1].replace("(", "").replace(")", "").strip()
            pat = verb + " " + gov
            if "," in verb:
                osad = verb.split(" ")
                v = osad[0].strip()
                v1 = osad[1].replace(",", "").strip()
                v2 = osad[2].replace(",", "").strip()
                if len(osad)>3:
                    print("rohkem osasid!")
                uus1.append((pat, v+" "+v1, case, verbobl, isik, koht, muu))
                uus1.append((pat, v+" "+v2, case, verbobl, isik, koht, muu))
            else:
                uus1.append((pat, verb, case, verbobl, isik, koht, muu))

    df1 = pd.DataFrame(uus1, columns=["pattern", "word", "phrase_case", "verbobl", "isik", "koht", "muu"])
    #df1.to_csv(f"isikumaarused_{maarus_status}.csv", sep=",", encoding="utf-8")
    
    verb_word, comp1, comp2, comp3 = process_verbs(df1)
    df1['verb_word'] = verb_word
    df1['verb_compound'] = comp1
    #df1['compound_prt2'] = comp2
    #df1['compound_prt3'] = comp3
    df1['phrase_nr'] = 1
    
    df1.insert(0, 'pat_id', range(1, 1 + len(df1)))
    df1.insert(7, 'adp', '')
    df1.insert(8, 'deprel', deprel)
    df1.insert(9, 'inf_verb', '')

    df2 = df1.drop(columns=['word'])
    
    df2.to_csv(f"../example_data/verb_patterns.csv", sep=",", encoding="utf-8", index = False)
    
    return df2

In [20]:
df = process_isikumaarused(MAARUS_STATUS, DEPREL)

In [21]:
df = pd.read_csv(f"../example_data/verb_patterns.csv", sep=",", encoding="utf-8")
df = df[~df["verb_word"].isna()]
df = df.fillna('')

In [22]:
# üheliikmelised mustrid
df

,pat_id,pattern,phrase_case,verbobl,isik,koht,adp,deprel,inf_verb,muu,verb_word,verb_compound,phrase_nr
0,1,saama kellelt/millelt,abl,saama - abl (kellelt/millelt),vahel,vahel,,obl,,mitte kunagi,saama,,1
1,2,tulema kellelt/millelt,abl,tulema - abl (kellelt/millelt),vahel,vahel,,obl,,mitte kunagi,tulema,,1
2,3,küsima kellelt/millelt,abl,küsima - abl (kellelt/millelt),alati,mitte kunagi,,obl,,mitte kunagi,küsima,,1
3,4,nõudma kellelt/millelt,abl,nõudma - abl (kellelt/millelt),alati,mitte kunagi,,obl,,mitte kunagi,nõudma,,1
4,5,võtma kellelt/millelt,abl,võtma - abl (kellelt/millelt),vahel,vahel,,obl,,mitte kunagi,võtma,,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10600,10601,musitseerima kelles/milles,in,musitseerima - in (kelles/milles),mitte kunagi,alati,,obl,,muu,musitseerima,,1
10601,10602,kõigutama kelles/milles,in,kõigutama - in (kelles/milles),mitte kunagi,mitte kunagi,,obl,,muu,kõigutama,,1
10602,10603,kätlema kelles/milles,in,kätlema - in (kelles/milles),mitte kunagi,alati,,obl,,muu,kätlema,,1
10603,10604,kõmmutama kelles/milles,in,kõmmutama - in (kelles/milles),mitte kunagi,alati,,obl,,mitte kunagi,kõmmutama,,1


### Create patterns table in database

In [23]:
cur.execute("""DROP TABLE IF EXISTS {name}""".format(name=PATTERN_TABLE_NAME))

# andmebaasi tabelite loomine
cur.execute(
    """CREATE TABLE {tablename}
    (pat_id INTEGER PRIMARY KEY, pattern TEXT, phrase_case TEXT, verb_word TEXT, verb_compound TEXT, phrase_nr INT, adp TEXT, deprel TEXT, inf_verb TEXT)
    """.format(tablename=PATTERN_TABLE_NAME)
)

### Fill table with data

In [24]:
%%time

insert = f"INSERT INTO {PATTERN_TABLE_NAME} "

for idx, row in df.iterrows():
    cur.execute(insert + """
    (pat_id, pattern, phrase_case, verb_word, verb_compound, phrase_nr, adp, deprel, inf_verb) 
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);""", 
    (row["pat_id"], row['pattern'], row['phrase_case'], row['verb_word'], row['verb_compound'], row['phrase_nr'], row['adp'], row['deprel'], row['inf_verb']))
    
    con.commit()

CPU times: user 2.8 s, sys: 3.2 s, total: 6 s
Wall time: 39.6 s


### Create semantic_annotations table in database

In [40]:
cur.execute("""DROP TABLE IF EXISTS {name}""".format(name=SEMANTIC_ANNOTATIONS_TABLE))

# andmebaasi tabelite loomine
cur.execute(
    """CREATE TABLE {tablename}
    (pattern_id INT,phrase_nr INT, semantic_role TEXT, certainty TEXT)
    """.format(tablename=SEMANTIC_ANNOTATIONS_TABLE)
)

### Fill table with data

In [41]:
%%time
for idx, row in df.iterrows():

    pat_id = row["pat_id"]
    phrase_nr = row["phrase_nr"]
    
    for role in ["isik", "koht", "muu"]:
        certain = row[role]

        insert = f"INSERT INTO {SEMANTIC_ANNOTATIONS_TABLE} "
        cur.execute(insert + """
            (pattern_id, phrase_nr, semantic_role, certainty) 
            VALUES (?, ?, ?, ?);""", 
            (pat_id, phrase_nr, role, certain)
        )
    
        con.commit()

CPU times: user 5.35 s, sys: 8.88 s, total: 14.2 s
Wall time: 1min 31s


### Check table contents

In [42]:
query = """SELECT * from {new_table} limit 5""".format(new_table = PATTERN_TABLE_NAME)
source = pd.read_sql_query(query, con)
source

,pat_id,pattern,phrase_case,verb_word,verb_compound,phrase_nr,adp,deprel,inf_verb
0,1,saama kellelt/millelt,abl,saama,,1,,obl,
1,2,tulema kellelt/millelt,abl,tulema,,1,,obl,
2,3,küsima kellelt/millelt,abl,küsima,,1,,obl,
3,4,nõudma kellelt/millelt,abl,nõudma,,1,,obl,
4,5,võtma kellelt/millelt,abl,võtma,,1,,obl,


In [45]:
query = """SELECT count(*) from {new_table} limit 12""".format(new_table = PATTERN_TABLE_NAME)
source = pd.read_sql_query(query, con)
source

,count(*)
0,10515


In [43]:
query = """SELECT * from {new_table} limit 12""".format(new_table = SEMANTIC_ANNOTATIONS_TABLE)
source = pd.read_sql_query(query, con)
source

,pattern_id,phrase_nr,semantic_role,certainty
0,1,1,isik,vahel
1,1,1,koht,vahel
2,1,1,muu,mitte kunagi
3,2,1,isik,vahel
4,2,1,koht,vahel
5,2,1,muu,mitte kunagi
6,3,1,isik,alati
7,3,1,koht,mitte kunagi
8,3,1,muu,mitte kunagi
9,4,1,isik,alati


In [44]:
query = """SELECT count(*) from {new_table} limit 12""".format(new_table = SEMANTIC_ANNOTATIONS_TABLE)
source = pd.read_sql_query(query, con)
source

,count(*)
0,31545


In [46]:
con.close()